In [1]:
import sys
sys.path.append(r"C:\Users\hjia9\Documents\GitHub\data-analysis")
sys.path.append(r"C:\Users\hjia9\Documents\GitHub\data-analysis\ucla-lapd")

import math
import numpy as np
import h5py
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d, gaussian_filter
from scipy.signal import find_peaks, savgol_filter

import read_hdf5 as rh
from bapsflib import lapd


%matplotlib qt
plt.rcParams.update({'font.size': 16})

In [2]:
ifn = r"D:\data\LAPD\Mar26-data\11-lp-sweep-xyplane-bias.hdf5"

with lapd.File(ifn) as f:
    rh.show_info(f)
    adc, digi_dict = rh.read_digitizer_config(f)
    pos_array, xpos, ypos, zpos, npos, nshot = rh.read_bmotion_probe_motion(f)

11-lp-sweep-xyplane-bias.hdf5 Overview
Generated by bapsflib (v0.0.0)
Generated date: 3/15/2026 9:16:43 PM


Filename:     11-lp-sweep-xyplane-bias.hdf5
Abs. Path:    D:\data\LAPD\Mar26-data\11-lp-sweep-xyplane-bias.hdf5
LaPD version: 1.2
Investigator: Jia Han
Run Date:     3/11/2026 9:01:59 PM

Exp. and Run Structure:
  (set)  Electrode_Biasing
  (exp)  +-- mar2026-jia
  (run)  |   +-- 11-lp-sweep-xyplane-bias

Run Description:
    Recording langmuir probe sweep signal with bias; same condition as run09.
    Most diagnostic and bias settings same as feb2026 experiments.
    
    
    LAPD B field:
    Black magnets at south: 888 A (PS12, 13),
    Yellow & Purple magnets: configured for uniform 800 G
    Except Yellow magnet PS 3, 4: 0 kG
    Black magnets at north: 0 A (PS11)
    (1600G-Source: 800 G: Bulk, 0 G: north end)
    
    South LaB6 source:
    He plasma, Discharge PS Voltage: 80 V, discharge current: 4.3 kA
    65 V cathode-anode voltage,   1/3 Hz rep rate 
    Heater: 34.6

In [4]:
digi_dict

{1: [7, 8], 2: [4, 5, 6, 7]}

In [8]:
with lapd.File(ifn) as f:
    data, tarr = rh.read_data(f, 4, 5, index_arr=slice(npos*nshot), adc=adc)
    Vsweep = data['signal'].reshape((npos, nshot, -1)) * 100 # X50 probe
    Vsweep = np.mean(Vsweep, axis=1)

    data, tarr = rh.read_data(f, 4, 4, index_arr=slice(npos*nshot), adc=adc)
    Isweep = data['signal'].reshape((npos, nshot, -1)) / 7.2 # V=I * 7.2 Ohm

# data, tarr = rh.read_data(f, bd, 4, index_arr=slice(st_ind,st_ind+npos*nshot), adc=adc, control=[('6K Compumotor', 1)])
# IsweepR_dic[bd] = data['signal'].reshape((npos, nshot, -1)) * cal_fac[bd-1] / A[3]

c:\Users\hjia9\Documents\GitHub\data-analysis\.venv\Lib\site-packages\bapsflib-0.0.0-py3.11.egg\bapsflib\_hdf\utils\hdfreaddata.py:508: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  data = np.empty(shape, dtype=dtype)
c:\Users\hjia9\Documents\GitHub\data-analysis\.venv\Lib\site-packages\bapsflib-0.0.0-py3.11.egg\bapsflib\_hdf\utils\hdfreaddata.py:508: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  data = np.empty(shape, dtype=dtype)


In [26]:
plt.plot(tarr, Vsweep[0])

In [21]:
def find_sweep_indices(V, threshold_frac=0.15, min_duration=50, merge_gap=50):
    """
    Extracts the start and stop indices for triangle voltage sweeps of any duration.
    
    Parameters:
        V (numpy.ndarray): The 1D array of the voltage trace.
        threshold_frac (float): The fraction of the maximum gradient to use as a cutoff threshold.
                                0.15 means 15% of the max slope.
        min_duration (int): The minimum number of points a sweep must last to not be considered noise.
        merge_gap (int): If a gap between "active" sweeping regions is smaller than this, 
                         they are merged. This is crucial because at the very tip of the 
                         triangle wave, the derivative momentarily hits 0 and could split the sweep.
                         
    Returns:
        start_t_ls (list): List of start indices for each sweep.
        stop_t_ls (list): List of stop indices for each sweep.
    """
    # 1. Calculate the absolute derivative (slope magnitude)
    dV = np.abs(np.gradient(V))
    
    # 2. Smooth the derivative slightly. This prevents noise spikes from 
    # dropping below the threshold and fragmenting the sweep.
    window = 10
    smoothed_dV = np.convolve(dV, np.ones(window)/window, mode='same')
    
    # 3. Create a boolean mask where the slope is significant (i.e., sweeping)
    threshold = np.max(smoothed_dV) * threshold_frac
    is_sweeping = smoothed_dV > threshold
    
    # 4. Find the edges (where the mask changes from False to True, and True to False)
    # We pad the mask with False at the ends to handle sweeps that touch the edges of the array
    padded_mask = np.concatenate(([False], is_sweeping, [False]))
    diffs = np.diff(padded_mask.astype(int))
    
    starts = np.where(diffs == 1)[0]
    stops = np.where(diffs == -1)[0]
    
    # 5. Filter and merge segments (closes the zero-crossing gap at the triangle apex)
    raw_starts, raw_stops = [], []
    
    for i in range(len(starts)):
        start = starts[i]
        stop = stops[i]
        
        # If this isn't the first segment, check if it's close enough to merge with the previous one
        if raw_stops and (start - raw_stops[-1]) < merge_gap:
            # Update the previous stop index to this new stop index
            raw_stops[-1] = stop
        else:
            # Add as a completely new sweep
            raw_starts.append(start)
            raw_stops.append(stop)
            
    # 6. Filter out very short segments (noise rejection)
    start_t_ls, stop_t_ls = [], []
    for start, stop in zip(raw_starts, raw_stops):
        if (stop - start) >= min_duration:
            start_t_ls.append(start)
            stop_t_ls.append(stop)
            
    return start_t_ls, stop_t_ls

In [23]:
start_t_ls, stop_t_ls = find_sweep_indices(Vsweep[0])
print(start_t_ls)

# def reshape_IV(Vsweep_arr, Isweep_arr):
#     # Initialize an empty list to store the chunks
#     I_chunks = []
#     V_chunks = []

#     # Loop through the pairs of start_t_ls and stop_t_ls
#     for start, stop in zip(start_t_ls, stop_t_ls):

#         I_chunks.append(Isweep_arr[:, :, start:stop])
#         V_chunks.append(Vsweep_arr[:, start:stop])

#     # Stack the list of chunks into a new array
#     Isweep_reshaped = np.stack(I_chunks, axis=2)
#     Vsweep_reshaped = np.stack(V_chunks, axis=1)

#     return Vsweep_reshaped, -Isweep_reshaped

[1267, 27516, 50018, 76268]
